# Lean 4 verification: spectral stability of strong ideal-gas shock layers

This notebook rebuilds and checks the Lean 4 formalization that accompanies the manuscript *Spectral stability for strong ideal-gas shock layers*. It depends only on Mathlib v4.34.1 and Lean v4.34.1. Run the cells from top to bottom. The sources are embedded in the notebook, so nothing needs to be uploaded or placed by hand. Section numbers below refer to the manuscript.

**Checked in Lean from first principles**

* The cancellation of the thermal mass coupling by the change of variables `h = ε + c u` (Section 4).
* The pointwise energy identity (A.1), as an exact identity using the profile relations.
* The explicit limiting profile of Lemma 2.1, which also shows that the profile hypotheses can be satisfied.
* The elimination of the internal energy, the derivatives of all coefficients along the profile, and the invariant slope interval of Lemma 2.2.
* The scalar bounds and coercivity estimates of Lemma 6.1 and Proposition 6.2 (reference line) and of Lemma 7.1 and Proposition 7.2 (explicit region), including the decomposition (7.4). The polynomial inequalities are proved by exact Bernstein expansions checked with `ring` and `positivity`.
* The vanishing of the energy flux (Lemma 8.1) and the exclusion of nonzero frequencies (Proposition 8.2).
* The zero-frequency argument (Proposition 9.1), including the downstream eigenvector and covector algebra.
* The compactness argument of Section 10 that passes to sufficiently strong finite shocks, uniformly over the closed band, and its link to Theorems 1.1 and 1.2 through the Evans function.

**Entering as explicit hypotheses.** Results of Humpherys, Lyng, and Zumbrun (Arch. Ration. Mech. Anal., 2009) that are outside the present scope of Mathlib. These are the existence of the limiting profile with its endstate limits and a positive downstream slope limit, the exponential decay of admissible solutions, the continued outgoing space at zero frequency, the saddle structure of the downstream rest point, and, for finite shocks, the continuity, high-frequency bound, and characterization of zeros of the integrated Evans function. They appear as hypotheses of the theorems, never as axioms.

The audit cell confirms that the sources contain no `sorry` and that every main result relies only on Lean's three standard axioms.


In [ ]:
# 1. Install elan (the Lean toolchain manager). About 30 s.
import subprocess, sys, time, os
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
def run(cmd, cwd="/content/StrongShock", check=True, show_all=False):
    """Run a shell command, streaming its output."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if show_all or not line.startswith(("warning", "Note:", "Hint:", "  [apply]", "  ")):
            print(line, end="")
    p.wait()
    print(f"[exit {p.returncode}, {time.time()-t0:.0f} s]")
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return "".join(lines)

if subprocess.run("command -v elan", shell=True).returncode != 0:
    run("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -o /tmp/elan-init.sh"
        " && bash /tmp/elan-init.sh -y --default-toolchain none", cwd="/tmp")
_ = run("elan --version", cwd="/tmp")


In [ ]:
# 2. Unpack the formalization (embedded as base64; nothing to upload).
import base64, io, tarfile, os
PAYLOAD = "..."
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz") as tf:
    try:
        tf.extractall("/content", filter="data")
    except TypeError:
        tf.extractall("/content")
n = 0
for dp, dn, fn in sorted(os.walk("/content/StrongShock")):
    for f in sorted(fn):
        n += 1
print(n, "files unpacked into /content/StrongShock")
print(open("/content/StrongShock/README.md").read())


In [ ]:
# 3. Fetch Mathlib v4.34.1 (pinned in lake-manifest.json) and its prebuilt binary cache.
_ = run("lake update")
_ = run("lake exe cache get")


In [ ]:
# 4. Build and check every proof. Timing is dominated by the polynomial expansions;
#    expect roughly 10-25 minutes depending on the runtime's cores.
log = run("lake build", check=False)
assert "error:" not in log and "Build completed successfully" in log, "build failed"
print("BUILD OK")


In [ ]:
# 5. Audit: no sorry/admit/axiom in the sources, and the axioms behind the main theorems.
import re, glob
srcs = glob.glob("/content/StrongShock/StrongShock/**/*.lean", recursive=True) + ["/content/StrongShock/StrongShock.lean"]
bad = [f for f in srcs if re.search(r"\bsorry\b|\badmit\b|^\s*axiom\s", open(f).read(), re.M)]
print("source files with sorry/admit/axiom:", bad if bad else "none")
out = run("lake env lean Check.lean", check=True, show_all=True)
std = "[propext, Classical.choice, Quot.sound]"
lines = [l for l in out.splitlines() if "depends on axioms" in l]
ok = (not bad) and len(lines) > 0 and all(l.endswith(std) for l in lines)
print("\nRESULT:", f"{len(lines)} main results checked; all use only {std}" if ok else "CHECK FAILED")


In [ ]:
# 6. (Optional) Regenerate the Bernstein expansions and bridge identities with the bundled
#    Python tools and confirm they are byte-identical to the files that were just checked.
_ = run("rm -rf /tmp/regen && mkdir -p /tmp/regen/StrongShock && SS_ROOT=/tmp/regen python3 gen_all.py",
    cwd="/content/StrongShock/tools")
_ = run("diff -r /tmp/regen/StrongShock/Bernstein StrongShock/Bernstein && diff -r /tmp/regen/StrongShock/Bridge StrongShock/Bridge"
    " && echo GENERATED FILES REPRODUCED")
